In [ ]:
# 1. 환경 설정 (Drive 마운트 + 패키지 설치)
import os, subprocess, sys, shutil

# ── Google Drive 마운트 ──────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# ── 경로 설정 ────────────────────────────────────────────────────
DRIVE_DIR  = "/content/drive/MyDrive/dacon"   # Drive (영구)
WORK_DIR   = "/content/dacon"                 # 로컬 (빠른 I/O, 휘발성)
LOCAL_DATA = os.path.join(WORK_DIR, "data")

for d in [DRIVE_DIR,
          os.path.join(DRIVE_DIR, "checkpoints"),
          os.path.join(DRIVE_DIR, "submissions"),
          WORK_DIR, LOCAL_DATA]:
    os.makedirs(d, exist_ok=True)

# ── GPU 확인 ─────────────────────────────────────────────────────
import torch
if torch.cuda.is_available():
    gpu  = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu} ({vram:.1f} GB)")
else:
    print("GPU 없음! 런타임 > 런타임 유형 변경 > GPU 선택")

# ── 디스크 여유 ──────────────────────────────────────────────────
free = shutil.disk_usage('/content').free / 1e9
print(f"로컬 디스크 여유: {free:.1f} GB")

# ── 패키지 설치 ──────────────────────────────────────────────────
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "timm>=1.0.20", "albumentations>=1.4.0",
    "opencv-python-headless", "scikit-learn", "pandas", "tqdm",
])
print("패키지 설치 완료")

In [ ]:
# 2. 소스 파일 복사 + 심볼릭 링크 (체크포인트 -> Drive)
import os, shutil

DRIVE_DIR = "/content/drive/MyDrive/dacon"
WORK_DIR  = "/content/dacon"

# ── 소스 파일: Drive → Local ─────────────────────────────────────
SRC_FILES = ["models.py", "datasets.py", "train.py", "inference.py"]
missing = []
for f in SRC_FILES:
    src = os.path.join(DRIVE_DIR, f)
    dst = os.path.join(WORK_DIR, f)
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f"  {f}")
    else:
        missing.append(f)
        print(f"  [MISSING] {f}")

if missing:
    print(f"\n다음 파일을 Drive에 업로드하세요: {DRIVE_DIR}/")
    for f in missing:
        print(f"  - {f}")

# ── checkpoints / submissions -> Drive 심볼릭 링크 ──────────────
# 세션 끊겨도 Drive에 남아있으므로 안전!
for name in ["checkpoints", "submissions"]:
    link_path  = os.path.join(WORK_DIR, name)
    drive_path = os.path.join(DRIVE_DIR, name)
    os.makedirs(drive_path, exist_ok=True)

    if os.path.islink(link_path):
        os.unlink(link_path)
    elif os.path.isdir(link_path):
        shutil.rmtree(link_path)

    os.symlink(drive_path, link_path)
    print(f"  {name}/ -> Drive (영구 저장)")

# ── 기존 체크포인트 확인 ─────────────────────────────────────────
ckpt_dir = os.path.join(DRIVE_DIR, "checkpoints")
ckpts = [f for f in os.listdir(ckpt_dir) if f.endswith('.pth')]
if ckpts:
    print(f"\nDrive 체크포인트 {len(ckpts)}개 발견 (--resume 으로 이어하기):")
    for c in sorted(ckpts):
        mb = os.path.getsize(os.path.join(ckpt_dir, c)) / 1e6
        tag = "[resume]" if "ckpt" in c else "[best]"
        print(f"  {tag} {c} ({mb:.0f} MB)")
else:
    print("\n체크포인트 없음 — 새로 학습 시작")

In [ ]:
# 3. 대회 데이터 추출
import os, shutil, zipfile
import pandas as pd

DRIVE_DIR  = "/content/drive/MyDrive/dacon"
WORK_DIR   = "/content/dacon"
LOCAL_DATA = os.path.join(WORK_DIR, "data")

train_csv = os.path.join(LOCAL_DATA, "open", "train.csv")

if os.path.exists(train_csv):
    n = len(pd.read_csv(train_csv))
    print(f"대회 데이터 이미 존재 (train: {n}개)")
else:
    # zip 파일 찾기
    zip_candidates = [
        os.path.join(DRIVE_DIR, "data.zip"),
        "/content/drive/MyDrive/data.zip",
        "/content/data.zip",
    ]
    zip_path = next((p for p in zip_candidates if os.path.exists(p)), None)

    if zip_path is None:
        print("data.zip을 찾을 수 없습니다!")
        print("다음 위치 중 하나에 업로드하세요:")
        for p in zip_candidates:
            print(f"  {p}")
    else:
        zip_gb  = os.path.getsize(zip_path) / 1e9
        free_gb = shutil.disk_usage('/content').free / 1e9
        print(f"data.zip: {zip_gb:.2f} GB | 여유: {free_gb:.1f} GB")

        # ── zip이 Drive에 있으면 로컬 복사 후 추출 -> 삭제 (속도 향상) ──
        if "/drive/" in zip_path and free_gb > zip_gb * 2.5:
            local_zip = "/content/_data_tmp.zip"
            print("Drive -> 로컬 복사 중...")
            shutil.copy2(zip_path, local_zip)
            extract_from = local_zip
        else:
            extract_from = zip_path
            local_zip = None

        print("추출 중...")
        with zipfile.ZipFile(extract_from, 'r') as zf:
            names = zf.namelist()
            if any(n.startswith("data/open/") for n in names):
                zf.extractall(WORK_DIR)
            elif any(n.startswith("open/") for n in names):
                zf.extractall(LOCAL_DATA)
            else:
                zf.extractall(LOCAL_DATA)

        # 로컬 임시 zip 삭제 (디스크 절약!)
        if local_zip and os.path.exists(local_zip):
            os.remove(local_zip)
            print("임시 zip 삭제 완료")

        # 검증
        for f in ["open/train.csv", "open/dev.csv", "open/sample_submission.csv"]:
            p = os.path.join(LOCAL_DATA, f)
            ok = "[OK]" if os.path.exists(p) else "[FAIL]"
            print(f"  {ok} {f}")

        for split in ["train", "dev", "test"]:
            d = os.path.join(LOCAL_DATA, "open", split)
            if os.path.isdir(d):
                n = len([x for x in os.listdir(d) if os.path.isdir(os.path.join(d, x))])
                print(f"  {split}/: {n}개")

        free_after = shutil.disk_usage('/content').free / 1e9
        print(f"\n추출 후 여유: {free_after:.1f} GB")

In [ ]:
# 4. ShapeStacks h=6 준비 (선택 — 없으면 Dacon 데이터만 pretrain)
import os, shutil, zipfile

DRIVE_DIR  = "/content/drive/MyDrive/dacon"
WORK_DIR   = "/content/dacon"
LOCAL_DATA = os.path.join(WORK_DIR, "data")

# Drive에 h=6 추출본 저장 -> 다음 세션에도 유지!
DRIVE_SS_H6 = os.path.join(DRIVE_DIR, "shapestacks_h6")
LOCAL_SS    = os.path.join(LOCAL_DATA, "shapestacks")

drive_rec = os.path.join(DRIVE_SS_H6, "shapestacks", "recordings")

# ── Case 1: Drive에 이미 추출되어 있음 -> 심볼릭 링크만 ────────
if os.path.exists(drive_rec):
    h6_dirs = [d for d in os.listdir(drive_rec) if "h=6" in d]
    print(f"Drive에 ShapeStacks h=6 이미 존재 ({len(h6_dirs)} scenarios)")

    if os.path.islink(LOCAL_SS):
        os.unlink(LOCAL_SS)
    elif os.path.isdir(LOCAL_SS):
        shutil.rmtree(LOCAL_SS)
    os.symlink(DRIVE_SS_H6, LOCAL_SS)
    print(f"  data/shapestacks -> Drive (재추출 불필요)")

else:
    # ── Case 2: zip에서 h=6만 선택 추출 -> Drive에 저장 ────────
    zip_candidates = [
        os.path.join(DRIVE_DIR, "shapestacks.zip"),
        "/content/drive/MyDrive/shapestacks.zip",
    ]
    zip_path = next((p for p in zip_candidates if os.path.exists(p)), None)

    if zip_path is None:
        print("ShapeStacks zip을 찾을 수 없습니다.")
        print("  -> Pretrain은 Dacon 데이터만으로 진행됩니다.")
        print(f"  -> 필요시 Drive에 업로드: {zip_candidates[0]}")
    else:
        zip_gb  = os.path.getsize(zip_path) / 1e9
        print(f"shapestacks.zip: {zip_gb:.1f} GB")

        # 핵심: h=6만 Drive에 추출 (최초 1회)
        # 전체 33GB 중 h=6만 ~11GB -> 디스크 절약
        # Drive에 추출 -> 다음 세션에도 심볼릭 링크로 즉시 사용
        print("\nh=6 시나리오만 Drive에 선택 추출 중...")
        print("  (최초 1회만 필요, 이후 세션에서는 자동 스킵)")

        os.makedirs(DRIVE_SS_H6, exist_ok=True)

        with zipfile.ZipFile(zip_path, 'r') as zf:
            all_names = zf.namelist()
            # h=6 시나리오 + splits 파일만 필터
            h6_names = [n for n in all_names
                        if "h=6" in n or "/splits/" in n]

            print(f"  전체: {len(all_names)} 파일")
            print(f"  h=6 + splits: {len(h6_names)} 파일")

            from tqdm.notebook import tqdm
            for name in tqdm(h6_names, desc="Extracting h=6"):
                zf.extract(name, DRIVE_SS_H6)

        # 심볼릭 링크
        if os.path.islink(LOCAL_SS):
            os.unlink(LOCAL_SS)
        elif os.path.isdir(LOCAL_SS):
            shutil.rmtree(LOCAL_SS)
        os.symlink(DRIVE_SS_H6, LOCAL_SS)

        # 통계
        if os.path.exists(drive_rec):
            dirs = [d for d in os.listdir(drive_rec) if "h=6" in d]
            stable = unstable = 0
            for d in dirs:
                is_unstable = False
                for part in d.split("-"):
                    if part.startswith("vcom=") or part.startswith("vpsf="):
                        try:
                            if int(part.split("=")[1]) > 0:
                                is_unstable = True
                        except ValueError:
                            pass
                if is_unstable:
                    unstable += 1
                else:
                    stable += 1
            print(f"\nh=6 추출 완료: {len(dirs)} scenarios (stable={stable}, unstable={unstable})")

In [ ]:
# 5. 설정 검증
import os

WORK_DIR  = "/content/dacon"
DRIVE_DIR = "/content/drive/MyDrive/dacon"

print("=" * 50)
print("  환경 검증")
print("=" * 50)

# 소스 파일
for f in ["models.py", "datasets.py", "train.py", "inference.py"]:
    ok = os.path.exists(os.path.join(WORK_DIR, f))
    print(f"  {'[OK]' if ok else '[FAIL]'} {f}")

# 데이터
tc = os.path.join(WORK_DIR, "data", "open", "train.csv")
print(f"  {'[OK]' if os.path.exists(tc) else '[FAIL]'} train.csv")

# ShapeStacks
ss = os.path.join(WORK_DIR, "data", "shapestacks", "shapestacks", "recordings")
if os.path.exists(ss):
    h6 = [d for d in os.listdir(ss) if "h=6" in d]
    print(f"  [OK] ShapeStacks h=6: {len(h6)} scenarios")
else:
    print(f"  [SKIP] ShapeStacks 없음 (Dacon only)")

# 체크포인트 -> Drive
ckpt_link = os.path.join(WORK_DIR, "checkpoints")
is_drive = os.path.islink(ckpt_link) and "drive" in os.readlink(ckpt_link)
print(f"  {'[OK]' if is_drive else '[FAIL]'} checkpoints/ -> Drive")

# submissions -> Drive
sub_link = os.path.join(WORK_DIR, "submissions")
is_drive2 = os.path.islink(sub_link) and "drive" in os.readlink(sub_link)
print(f"  {'[OK]' if is_drive2 else '[FAIL]'} submissions/ -> Drive")

# 기존 체크포인트
ckpt_dir = os.path.join(DRIVE_DIR, "checkpoints")
ckpts = [f for f in os.listdir(ckpt_dir) if f.endswith('.pth')]
resume_ckpts = [c for c in ckpts if "ckpt" in c]
best_ckpts   = [c for c in ckpts if "ckpt" not in c]

print()
if resume_ckpts:
    print(f"  Resume 체크포인트: {len(resume_ckpts)}개 -> 자동 이어하기!")
    for c in sorted(resume_ckpts):
        print(f"    {c}")
if best_ckpts:
    print(f"  Best 체크포인트: {len(best_ckpts)}개")
    for c in sorted(best_ckpts):
        print(f"    {c}")
if not ckpts:
    print("  체크포인트 없음 -> 새로 학습 시작")

import shutil
free = shutil.disk_usage('/content').free / 1e9
print(f"\n  디스크 여유: {free:.1f} GB")
print("=" * 50)

In [ ]:
# 6. 학습 설정
# 백본과 에폭 수를 설정하세요. 재접속 후에도 이 셀을 먼저 실행!
# 선택: "eva_giant", "dinov3_huge", "siglip_so400m", "eva02_large", "dinov2_large"

BACKBONE = "eva_giant"
PRETRAIN_EPOCHS = 15
FINETUNE_EPOCHS = 50
INCLUDE_DEV = True
USE_VIDEO = False

import os
os.chdir("/content/dacon")

print(f"백본: {BACKBONE}")
print(f"Pretrain: {PRETRAIN_EPOCHS} ep -> Finetune: {FINETUNE_EPOCHS} ep")
print(f"Dev 포함: {INCLUDE_DEV} | 비디오: {USE_VIDEO}")
print(f"작업 디렉토리: {os.getcwd()}")

In [ ]:
# 7. Stage 1: Pretrain (ShapeStacks h=6 + Dacon)
# 끊겨도 --resume 으로 자동 이어하기

import os
os.chdir("/content/dacon")
os.system(
    f"python train.py"
    f" --backbone {BACKBONE}"
    f" --stage pretrain"
    f" --pretrain_epochs {PRETRAIN_EPOCHS}"
    f" --grad_checkpointing"
    f" --resume"
    f" --num_workers 4"
)

In [ ]:
# 8. Stage 2: Finetune (5-Fold CV)
# 끊겨도 --resume 으로 자동 이어하기. Fold 단위로 체크포인트 저장.

import os
os.chdir("/content/dacon")

cmd = (
    f"python train.py"
    f" --backbone {BACKBONE}"
    f" --stage finetune"
    f" --finetune_epochs {FINETUNE_EPOCHS}"
    f" --grad_checkpointing"
    f" --resume"
    f" --num_workers 4"
)
if INCLUDE_DEV:
    cmd += " --include_dev"
if USE_VIDEO:
    cmd += " --use_video"

print(f"CMD: {cmd}")
os.system(cmd)

In [ ]:
# (선택) 단일 Fold 테스트 — 빠른 검증용

FOLD = 0
QUICK_EPOCHS = 10

import os
os.chdir("/content/dacon")
os.system(
    f"python train.py"
    f" --backbone {BACKBONE}"
    f" --stage finetune"
    f" --finetune_epochs {QUICK_EPOCHS}"
    f" --fold {FOLD}"
    f" --grad_checkpointing"
    f" --resume"
    f" --num_workers 4"
    f" --include_dev"
)

In [ ]:
# 9. 추론 + 제출 파일 생성

import os, glob
import pandas as pd
os.chdir("/content/dacon")

os.system(
    f"python inference.py"
    f" --backbones {BACKBONE}"
    f" --tta"
    f" --temperature 1.0"
)

# 결과 확인
subs = sorted(glob.glob("/content/dacon/submissions/*.csv"))
if subs:
    latest = subs[-1]
    df = pd.read_csv(latest)
    print(f"\n최신 제출: {os.path.basename(latest)}")
    print(f"  샘플 수: {len(df)}")
    print(f"  unstable_prob: mean={df['unstable_prob'].mean():.4f}, std={df['unstable_prob'].std():.4f}")
    print(df.head())
    print(f"\n제출 파일은 Drive에 저장됨: /content/drive/MyDrive/dacon/submissions/")
else:
    print("제출 파일이 생성되지 않았습니다. 체크포인트가 있는지 확인하세요.")

In [ ]:
# 10. Dev 검증

import os
os.chdir("/content/dacon")
os.system(
    f"python inference.py"
    f" --backbones {BACKBONE}"
    f" --tta"
    f" --validate"
)

In [ ]:
# 11. 다중 백본 앙상블 파이프라인

BACKBONES = ["eva_giant", "dinov3_huge", "siglip_so400m"]
PRETRAIN_EP = 10
FINETUNE_EP = 40

import os
os.chdir("/content/dacon")

for bk in BACKBONES:
    print(f"\n{'='*60}")
    print(f"  Pretrain: {bk}")
    print(f"{'='*60}")
    os.system(
        f"python train.py --backbone {bk} --stage pretrain"
        f" --pretrain_epochs {PRETRAIN_EP}"
        f" --grad_checkpointing --resume --num_workers 4"
    )

    print(f"\n{'='*60}")
    print(f"  Finetune: {bk}")
    print(f"{'='*60}")
    os.system(
        f"python train.py --backbone {bk} --stage finetune"
        f" --finetune_epochs {FINETUNE_EP}"
        f" --include_dev --grad_checkpointing --resume --num_workers 4"
    )

# 앙상블 추론
bk_str = " ".join(BACKBONES)
os.system(f"python inference.py --backbones {bk_str} --tta")

print("\n다중 백본 앙상블 완료!")
print("제출 파일: /content/drive/MyDrive/dacon/submissions/")